## Install Dependencies

Installs the required Python packages for this notebook:
- **azure-ai-evaluation[redteam]** – Azure AI Evaluation SDK with red-teaming capabilities for scanning model endpoints.
- **azure-identity** – Azure SDK credential library for authentication.
- **httpx** – HTTP client used by the REST callback to invoke the online endpoint.
- **dotenv** – Utility for loading environment variables from a `.env` file.
- **azure-ai-ml** – Azure Machine Learning SDK v2 for managing endpoints and retrieving access keys.
- **azure-keyvault-secrets** – Azure SDK library for reading secrets from Key Vault.

In [ ]:
%pip install azure-ai-evaluation[redteam] azure-identity httpx dotenv 
%pip install azure-ai-ml azure-identity azure-keyvault-secrets

## Configure Azure ML Client and Load Secrets from Key Vault

Initialises the Azure ML client using `DefaultAzureCredential` and the workspace config file, then retrieves all deployment configuration values from Azure Key Vault. Each secret (model ID, model name, scoring script, instance type, name prefixes, subscription, resource group, workspace, and Azure AI Foundry deployment details) can be overridden at runtime via environment variables, allowing the notebook to be driven from a CI/CD pipeline without modifying source code. The resolved values are printed for verification.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import  os

AZURE_ML_MODEL_EMPTY_ID_SECRET_NAME="AZURE-ML-MODEL-EMPTY-ID"
AZURE_ML_MODEL_EMPTY_NAME_SECRET_NAME="AZURE-ML-MODEL-EMPTY-NAME"
AZURE_ML_MODEL_EMPTY_SCORE_SCRIPT_SECRET_NAME="AZURE-ML-MODEL-EMPTY-SCORE-SCRIPT"
AZURE_ML_MODEL_EMPTY_INSTANCE_TYPE_SECRET_NAME="AZURE-ML-MODEL-EMPTY-INSTANCE-TYPE" 

AZURE_ML_MODEL_PREFIX_SECRET_NAME="AZURE-ML-MODEL-PREFIX" 
AZURE_ML_ENDPOINT_PREFIX_SECRET_NAME="AZURE-ML-ENDPOINT-PREFIX" 
AZURE_ML_ENV_PREFIX_SECRET_NAME="AZURE-ML-ENV-PREFIX" 
AZURE_ML_SUBSCRIPTION_ID_SECRET_NAME="AZURE-ML-SUBSCRIPTION-ID" 
AZURE_ML_RESOURCE_GROUP_SECRET_NAME="AZURE-ML-RESOURCE-GROUP" 
AZURE_ML_WORKSPACE_NAME_SECRET_NAME="AZURE-ML-WORKSPACE-NAME"

AZURE_MODEL_DEPLOYMENT_NAME_SECRET_NAME="AZURE-ML-MODEL-DEPLOYMENT-NAME" 
AZURE_MODEL_DEPLOYMENT_URI_SECRET_NAME="AZURE-ML-MODEL-DEPLOYMENT-URI" 
AZURE_MODEL_DEPLOYMENT_KEY_SECRET_NAME="AZURE-ML-MODEL-DEPLOYMENT-KEY" 
AZURE_MODEL_DEPLOYMENT_MODEL_API_VERSION_SECRET_NAME="AZURE-ML-MODEL-DEPLOYMENT-MODEL-API-VERSION" 
AZURE_MODEL_DEPLOYMENT_MODEL_NAME_SECRET_NAME="AZURE-ML-MODEL-DEPLOYMENT-MODEL-NAME" 
AZURE_FOUNDRY_PROJECT_ENDPOINT_SECRET_NAME="AZURE-FOUNDRY-PROJECT-ENDPOINT"

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)
keyvault_name = ml_client.workspaces.get(ml_client.workspace_name).key_vault.split('/')[-1]
kv_url = f"https://{keyvault_name}.vault.azure.net/"



def readSecret(
    key_vault_url: str,
    secret_name: str
) -> str:
    """
    read secret in keyvault .

    Args:
        key_vault_url:  Key Vault url.
        secret_name:    Name of the secret
    Returns:
        secret value (str).
    """        
    kv_client = SecretClient(vault_url=key_vault_url, credential=credential)
    secret_value = kv_client.get_secret(secret_name).value
    return secret_value

global_model_id = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_ID_SECRET_NAME)
global_model_id=os.getenv("AZURE_ML_MODEL_ID",default=global_model_id)
print(f"global_model_id:{global_model_id}")

local_model_name = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_NAME_SECRET_NAME)
local_model_name=os.getenv("AZURE_ML_MODEL_NAME",default=local_model_name)
print(f"local_model_name:{local_model_name}")

global_instance_type = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_INSTANCE_TYPE_SECRET_NAME)
global_instance_type=os.getenv("AZURE_ML_INSTANCE_TYPE",default=global_instance_type)
print(f"global_instance_type:{global_instance_type}")

global_score_script = readSecret(kv_url,AZURE_ML_MODEL_EMPTY_SCORE_SCRIPT_SECRET_NAME)
global_score_script=os.getenv("AZURE_ML_SCORING_SCRIPT",default=global_score_script)
print(f"global_score_script:{global_score_script}")


prefix = readSecret(kv_url,AZURE_ML_MODEL_PREFIX_SECRET_NAME)
global_model_name = f"{prefix}-{local_model_name}"
global_model_name=os.getenv("AZURE_ML_MODEL_NAME",default=global_model_name)
print(f"global_model_name:{global_model_name}")

prefix = readSecret(kv_url,AZURE_ML_ENDPOINT_PREFIX_SECRET_NAME)
global_endpoint_name = f"{prefix}-{local_model_name}"
global_endpoint_name=os.getenv("AZURE_ML_ENDPOINT_NAME",default=global_endpoint_name)
print(f"global_endpoint_name:{global_endpoint_name}")

prefix = readSecret(kv_url,AZURE_ML_ENV_PREFIX_SECRET_NAME)
global_env_name = f"{prefix}-{local_model_name}"
global_env_name=os.getenv("AZURE_ML_ENV_NAME",default=global_env_name)
print(f"global_env_name:{global_env_name}")


global_subscription_id = readSecret(kv_url,AZURE_ML_SUBSCRIPTION_ID_SECRET_NAME)
global_subscription_id=os.getenv("AZURE_ML_SUBSCRIPTION_ID",default=global_subscription_id)
print(f"global_subscription_id:{global_subscription_id}")
    
global_resource_group_name = readSecret(kv_url,AZURE_ML_RESOURCE_GROUP_SECRET_NAME)
global_resource_group_name=os.getenv("AZURE_ML_RESOURCE_GROUP",default=global_resource_group_name)
print(f"global_resource_group_name:{global_resource_group_name}")

global_workspace_name = readSecret(kv_url,AZURE_ML_WORKSPACE_NAME_SECRET_NAME)
global_workspace_name=os.getenv("AZURE_ML_WORKSPACE_NAME",default=global_workspace_name)
print(f"global_workspace_name:{global_workspace_name}")

global_model_deployment_name = readSecret(kv_url,AZURE_MODEL_DEPLOYMENT_NAME_SECRET_NAME)
global_model_deployment_name=os.getenv("AZURE_ML_MODEL_DEPLOYMENT_NAME",default=global_model_deployment_name)
print(f"global_model_deployment_name:{global_model_deployment_name}")

global_model_deployment_uri = readSecret(kv_url,AZURE_MODEL_DEPLOYMENT_URI_SECRET_NAME)
global_model_deployment_uri=os.getenv("AZURE_ML_MODEL_DEPLOYMENT_URI",default=global_model_deployment_uri)
print(f"global_model_deployment_uri:{global_model_deployment_uri}")

global_model_deployment_key = readSecret(kv_url,AZURE_MODEL_DEPLOYMENT_KEY_SECRET_NAME)
global_model_deployment_key=os.getenv("AZURE_ML_MODEL_DEPLOYMENT_KEY",default=global_model_deployment_key)
print(f"global_model_deployment_key:{global_model_deployment_key}")

global_model_deployment_model_api_version = readSecret(kv_url,AZURE_MODEL_DEPLOYMENT_MODEL_API_VERSION_SECRET_NAME)
global_model_deployment_model_api_version=os.getenv("AZURE_ML_MODEL_DEPLOYMENT_MODEL_API_VERSION",default=global_model_deployment_model_api_version)
print(f"global_model_deployment_model_api_version:{global_model_deployment_model_api_version}")

global_model_deployment_model_name = readSecret(kv_url,AZURE_MODEL_DEPLOYMENT_MODEL_NAME_SECRET_NAME)
global_model_deployment_model_name=os.getenv("AZURE_ML_MODEL_DEPLOYMENT_MODEL_NAME",default=global_model_deployment_model_name)
print(f"global_model_deployment_model_name:{global_model_deployment_model_name}")


global_foundry_project_endpoint = readSecret(kv_url,AZURE_FOUNDRY_PROJECT_ENDPOINT_SECRET_NAME)
global_foundry_project_endpoint=os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT",default=global_foundry_project_endpoint)
print(f"global_foundry_project_endpoint:{global_foundry_project_endpoint}")


## Define Helper Functions

Defines the utility functions used to configure the red-team scan target and parse scan arguments:

- **`build_azure_openai_config()`** – Builds the Azure OpenAI model configuration dict (endpoint, key, deployment name, API version) for use when targeting an Azure AI Foundry deployment directly.
- **`make_simple_rest_callback()`** – Creates a REST callback function that posts a `{"text": query}` payload to an Azure ML online endpoint and returns the `generated_text` field from the response. Supports bearer token authentication.
- **`to_risk_categories()`** / **`to_attack_strategies()`** / **`to_language()`** – Convert comma-separated string arguments to the corresponding SDK enum values.
- **`ensure_https()`** – Validates and upgrades the endpoint URL to HTTPS, since bearer token auth requires TLS.
- **`parse_args()`** – Defines and parses the CLI-style arguments that control the scan (mode, output path, scan name, objectives, risk categories, attack strategies, and endpoint URL).

In [ ]:
import os
import argparse
from typing import Optional, Dict
import argparse
from azure.identity import DefaultAzureCredential


def build_azure_openai_config() -> Dict[str, str]:
    # Supported target: model configuration dict (Azure OpenAI) [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    if global_model_deployment_key == "null":
        return {
            "azure_endpoint": global_model_deployment_uri,
            "credential": DefaultAzureCredential(),
            "azure_deployment": global_model_deployment_name,
            "api_version": global_model_deployment_model_api_version
        }
    else:
        return {
            "azure_endpoint": global_model_deployment_uri,
            "api_key": global_model_deployment_key,
            "azure_deployment": global_model_deployment_name,
            "api_version": global_model_deployment_model_api_version
        }

def make_simple_rest_callback(endpoint_url: str, bearer_token: Optional[str] = None):
    """
    Suggestion (not from docs): implement calling your Azure ML online endpoint (or any REST endpoint).
    The RedTeam SDK supports a simple callback signature: (query: str) -> str [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    """
    import httpx

    headers = {"Content-Type": "application/json"}
    if bearer_token:
        print("Using bearer token for authentication in REST callback")
        headers["Authorization"] = f"Bearer {bearer_token}"

    def callback(query: str) -> str:
        payload = {
                    "text": query
                }

        r = httpx.post(endpoint_url, json=payload, headers=headers, timeout=60)
        r.raise_for_status()
        data = r.json()

        return data["generated_text"]  if isinstance(data, dict) else str(data)

    return callback


def to_risk_categories(csv: str):
    mapping = {
        "Violence": RiskCategory.Violence,
        "HateUnfairness": RiskCategory.HateUnfairness,
        "Sexual": RiskCategory.Sexual,
        "SelfHarm": RiskCategory.SelfHarm,
        "ProtectedMaterial": RiskCategory.ProtectedMaterial,
        "CodeVulnerability": RiskCategory.CodeVulnerability,
        "UngroundedAttributes": RiskCategory.UngroundedAttributes,
    }
    return [mapping[x.strip()] for x in csv.split(",") if x.strip()]

def to_attack_strategies(csv: str):
    mapping = {
        "EASY": AttackStrategy.EASY,
        "MODERATE": AttackStrategy.MODERATE,
        "DIFFICULT": AttackStrategy.DIFFICULT,
    }
    return [mapping[x.strip()] for x in csv.split(",") if x.strip()]

def to_language(name: str):
    # SupportedLanguages is documented; list includes French, etc. [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    return getattr(SupportedLanguages, name, SupportedLanguages.English)

def ensure_https(url: str, name: str) -> str:
    print(f"url: {url}")
    if url.startswith("http://"):
        print(f"WARNING: {name} uses http:// — upgrading to https:// (bearer token auth requires TLS)")
        return "https://" + url[len("http://"):]
    if not url.startswith("https://"):
        raise ValueError(f"{name} must be an https:// URL, got: {url!r}")
    return url

def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--mode", choices=["azure_openai_config", "rest_callback"], required=True)
    p.add_argument("--output_path", default="redteam_scorecard.json")
    p.add_argument("--scan_name", default="RedTeam Scan")
    p.add_argument("--num_objectives", type=int, default=10)
    p.add_argument("--language", default="English")
    p.add_argument("--risk_categories", default="Violence,HateUnfairness,Sexual,SelfHarm")
    p.add_argument("--attack_strategies", default="EASY,MODERATE,DIFFICULT")
    p.add_argument("--rest_endpoint_url", default=None)
    return p.parse_args(argv)

## Run the Red-Team Scan

Defines the `main()` async function and immediately invokes it. It:

1. Resolves the Azure AI Foundry project URL and authenticates using `DefaultAzureCredential`.
2. Creates a `RedTeam` agent configured with the target risk categories and number of objectives.
3. Depending on `--mode`:
   - **`azure_openai_config`** – Targets the Azure AI Foundry model deployment directly via its OpenAI-compatible API.
   - **`rest_callback`** – Retrieves the Azure ML online endpoint's primary key, wraps the endpoint in a REST callback, and uses that as the scan target.
4. Runs the scan with the specified attack strategies and writes the scorecard JSON to `--output_path`.

The endpoint scoring URI is fetched dynamically from the Azure ML workspace before the scan begins.

In [ ]:
import os
import argparse
from typing import Optional, Dict
import argparse
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation.red_team import RedTeam, RiskCategory, AttackStrategy, SupportedLanguages  # [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model

async def main(argv=None):
    args = argv if isinstance(argv, argparse.Namespace) else parse_args(argv)

    # Foundry project identification supported as hub-style dict or AZURE_AI_PROJECT URL [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    print("Using AZURE_AI_PROJECT from environment")
    azure_ai_project = global_foundry_project_endpoint

    credential = DefaultAzureCredential()

    red_team_agent = RedTeam(
        azure_ai_project=azure_ai_project,
        credential=credential,
        risk_categories=to_risk_categories(args.risk_categories),
        num_objectives=args.num_objectives,
        language=to_language(args.language),
    )  # [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)

    attack_strategies = to_attack_strategies(args.attack_strategies)

    if args.mode == "azure_openai_config":
        target = build_azure_openai_config()
    else:
        if not args.rest_endpoint_url:
            raise ValueError("--rest_endpoint_url is required for rest_callback mode")

        subscription_id = global_subscription_id
        resource_group = global_resource_group_name
        workspace_name = global_workspace_name

        ml_client = MLClient(
            DefaultAzureCredential(),
            subscription_id,
            resource_group,
            workspace_name
        )

        creds = ml_client.online_endpoints.get_keys(
                        name=global_endpoint_name
                )

        # API keys (for auth_mode: key)
        access_token_str = creds.primary_key
        # token = credential.get_token("https://cognitiveservices.azure.com/.default")
        # access_token_str = token.token
        target = make_simple_rest_callback(args.rest_endpoint_url,access_token_str)

    result = await red_team_agent.scan(
        target=target,
        scan_name=args.scan_name,
        attack_strategies=attack_strategies,
        output_path=args.output_path,
    )  # [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)

    # The scan writes the scorecard JSON to output_path; result also includes row-level data. [1](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/foundry-classic/how-to/develop/run-scans-ai-red-teaming-agent.md)
    print(f"Saved scorecard to: {args.output_path}")
    return result

num_objectives = 2
categories = "Violence,HateUnfairness,Sexual,SelfHarm"
strategies = "EASY,MODERATE"
print("")
print(f"Launching tests on Model: {global_model_deployment_model_name}")
print(f"      Number: {num_objectives}")
print(f"  Categories: {categories}")
print(f"  Strategies: {strategies}")
args = parse_args([
    "--mode", "azure_openai_config",
    "--output_path", "outputs/redteam_scorecard_opt_350m_2",
    "--scan_name", "CI RedTeam Scan",
    "--num_objectives", str(num_objectives),
    "--risk_categories", categories,
    "--attack_strategies", strategies
])

res = await main(args)